<div dir="rtl" align="right">

# تحليلُ النمطِ التجريبيِّ \(EMD\)

**مجموعةُ البياناتِ**: PhysioNet Auditory EEG  
**القنواتُ**: P4, Cz, F8, T7  
**معدّلُ أخذِ العيناتِ**: 200 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

يُحلّلُ تحليلُ النمطِ التجريبيِّ (EMD) الإشارةَ إلى دوالَّ ذاتيّةٍ (IMFs) تَتدرّجُ من التردداتِ العاليةِ إلى المنخفضة. نَستخدمُ مكتبةَ PyEMD.

## المُخرجاتُ المُتوقّعةُ

- الإشارةُ الأصليّةُ في الأعلى
- خمسُ دوالَّ ذاتيّةٍ (IMFs) تَتدرّجُ من التردداتِ العاليةِ إلى المنخفضة
- IMF الأوّلُ يَلتقطُ التردداتِ العاليةَ (آثارٌ شائبةٌ)

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ | المعنى |
| --- | --- | --- |
| القناةُ | P4 | المنطقةُ الجداريةُ |
| العيّناتُ | 5000 | أولُ 25 ثانيةً |
| max_imf | 5 | الحدُّ الأقصى لِلدوالِّ |

</div>

<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>

In [ ]:
!pip install mne scikit-learn EMD-signal scipy numpy plotly wfdb


<div dir="rtl" align="right">

## 2. استنساخُ المستودعِ وتنزيلُ بياناتِ مُشاركٍ واحدٍ

نَنزّلُ مُشاركًا واحدًا فقط (`--subjects 1`) لتسريعِ التجربةِ في بيئةِ Colab.

</div>

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


<div dir="rtl" align="right">

## 3. تحميلُ إشارةِ EEG

نحمّلُ تسجيلَ المُشاركِ 1 في التجربةِ 1، الجلسةِ 2.

</div>

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
fs = 200

print(f'Channels: {ch_names}')
print(f'Signal length: {len(eeg_data)} samples ({len(eeg_data)/fs:.1f} seconds)')


<div dir="rtl" align="right">

## 4. تطبيقُ EMD

نُطبّقُ `EMD` من مكتبةِ PyEMD على أولِ 5000 عيّنةٍ من القناةِ P4.

</div>

In [ ]:
from PyEMD import EMD

n_plot = min(5000, len(eeg_data))
channel_data = eeg_data[:n_plot, 0]

emd = EMD()
imfs = emd(channel_data, max_imf=5)
print(f'Number of IMFs: {imfs.shape[0]}')


<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- IMF الأوّلُ يَحتوي على التردداتِ الأعلى
- الدوالُ اللاحقةُ تَحتوي على تردداتٍ أبطأ
- يُمكنُ حذفُ IMF الأوّل لِإزالةِ الآثارِ عاليةِ التردد


</div>

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

t_sec = np.arange(n_plot) / fs
n_imfs = min(imfs.shape[0], 5)

fig = make_subplots(rows=n_imfs + 1, cols=1, shared_xaxes=True,
                    subplot_titles=['Original'] + [f'IMF {i+1}' for i in range(n_imfs)])
fig.add_trace(go.Scatter(x=t_sec, y=channel_data, name='Original',
                         line=dict(color='black', width=0.5)), row=1, col=1)
for i in range(n_imfs):
    fig.add_trace(go.Scatter(x=t_sec, y=imfs[i], name=f'IMF {i+1}',
                             line=dict(color='blue', width=0.5)), row=i+2, col=1)
fig.update_layout(height=900, title_text='EMD Decomposition - Channel P4',
                  xaxis_title='Time (s)', showlegend=False)
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- EMD يُحلّلُ الإشارةَ إلى دوالَّ ذاتيّةٍ دونَ افتراضِ دوالَّ أساسيّة
- الدوالُ الأوّلُ تَلتقطُ التردداتِ العالية، والأخيرةُ المنخفضة
- يُمكنُ حذفُ دوالَّ مُحدّدةٍ لِإزالةِ الآثارِ الشائبة
- مُناسبٌ لِلإشاراتِ غيرِ الثابتةِ مثلِ EEG


</div>